# Titanic — Machine Learning from Disaster
### Random Forest workflow, parts A–F

**Session 3 · task 4 · `4.random_forest_titanic`**

This notebook answers parts A–F of the assignment in order. It runs end to end on Kaggle:
add the *Titanic* competition as **Input** (right panel → Add Input → Competitions → Titanic), then
**Save Version → Save & Run All**. `submission.csv` is written to `/kaggle/working`, from where you can
submit it to the leaderboard in one click.

| Part | What it does |
|---|---|
| A | `feature_importances_` of the forest, sorted, as a bar chart |
| B | Median age learned on the training subset only, applied to both subsets |
| C | Classification report + confusion matrix (`Did not survive` / `Survived`) |
| D | False positives vs false negatives, per-class recall |
| E | Decision trees with `max_depth` 1, 2, 4, 8 — depth, leaves, train accuracy, rules |
| F | Final model on all 891 rows → `submission.csv` for Kaggle |

Everything is deterministic: `random_state=42`.

## 0. Setup

The data lives in `/kaggle/input/titanic/` on Kaggle (or `data/` when run locally — the same 891-row
training set is also available through `seaborn.load_dataset("titanic")`).

All preprocessing sits **inside** a scikit-learn `Pipeline`, so imputation values are learned from
training rows only — this is what part B checks explicitly.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": "white",
})

RANDOM_STATE = 42
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)


def find_data_dir() -> Path:
    for cand in (Path("/kaggle/input/titanic"), Path("data"), Path("../data")):
        if (cand / "train.csv").exists():
            return cand
    hits = sorted(Path("/kaggle/input").rglob("train.csv"))
    if hits:
        return hits[0].parent
    raise FileNotFoundError("train.csv not found - add the Titanic competition as Input")


DATA = find_data_dir()
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print("data directory:", DATA.resolve())
print("train:", train.shape, "| test:", test.shape)
train.head()

### Features

| Feature | Type | Rationale |
|---|---|---|
| `Sex` | categorical | the strongest single signal ("women and children first") |
| `Title` (from `Name`) | categorical | Mr / Mrs / Miss / Master / Officer / Noble / Rare — sex + age + social role in one column |
| `Pclass` | categorical | ticket class as a proxy for wealth and deck location |
| `Age` | numeric | children were prioritised; median-imputed (part B) |
| `Fare` | numeric | price paid |
| `SibSp`, `Parch`, `FamilySize` | numeric | family structure on board |
| `IsAlone`, `HasCabin` | categorical | travelling alone; whether a cabin was recorded |

In [ ]:
TITLE_MAP = {
    "Mr": "Mr", "Mrs": "Mrs", "Miss": "Miss", "Master": "Master",
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Noble", "Countess": "Noble", "Dona": "Noble", "Don": "Noble",
    "Jonkheer": "Noble", "Sir": "Noble",
    "Capt": "Officer", "Col": "Officer", "Major": "Officer",
    "Dr": "Officer", "Rev": "Officer",
}


def engineer(df: pd.DataFrame) -> pd.DataFrame:
    '''Row-wise, deterministic features: nothing is learned from any subset here.'''
    d = df.copy()
    d["Title"] = d["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    d["Title"] = d["Title"].map(TITLE_MAP).fillna("Rare")
    d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
    d["IsAlone"] = (d["FamilySize"] == 1).astype(int)
    d["HasCabin"] = d["Cabin"].notna().astype(int)
    return d


train_fe, test_fe = engineer(train), engineer(test)

NUMERIC = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]
CATEGORICAL = ["Pclass", "Sex", "Embarked", "Title", "IsAlone", "HasCabin"]
FEATURES = NUMERIC + CATEGORICAL

X, y = train_fe[FEATURES], train_fe["Survived"]
X_test_kaggle = test_fe[FEATURES]

# one stratified 20 % hold-out, used for every "honest" evaluation below
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)


def build_pipeline(model) -> Pipeline:
    prep = ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), NUMERIC),
            ("cat", Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), CATEGORICAL),
        ],
        verbose_feature_names_out=False,
    )
    return Pipeline([("prep", prep), ("model", model)])


RF_PARAMS = dict(n_estimators=500, min_samples_leaf=2, max_features=0.4,
                 random_state=RANDOM_STATE, n_jobs=-1)


def rf(**kw):
    p = dict(RF_PARAMS); p.update(kw)
    return RandomForestClassifier(**p)


def save(fig, name):
    fig.savefig(OUT / name, bbox_inches="tight")
    plt.show()


print(f"train split {X_tr.shape} | validation split {X_va.shape} | survival rate {y.mean():.3f}")

## A. Which features matter most?

`feature_importances_` returns one value per **encoded** column, so `Sex` arrives as two columns
(`Sex_female`, `Sex_male`). The first chart sums those back to their source variable — that is the
readable version; the second keeps the raw per-column detail.

In [ ]:
pipe_a = build_pipeline(rf())
pipe_a.fit(X_tr, y_tr)
model_a = pipe_a.named_steps["model"]

names = [n.split("__")[-1] for n in pipe_a.named_steps["prep"].get_feature_names_out()]
imp = (pd.DataFrame({"feature": names, "importance": model_a.feature_importances_})
       .sort_values("importance", ascending=False).reset_index(drop=True))
imp["cum_importance"] = imp["importance"].cumsum()
display(imp.round(4))


def base_feature(name: str) -> str:
    for f in sorted(FEATURES, key=len, reverse=True):
        if name == f or name.startswith(f + "_"):
            return f
    return name


grouped = (imp.assign(base=imp["feature"].map(base_feature))
           .groupby("base", as_index=False)["importance"].sum()
           .sort_values("importance", ascending=False).reset_index(drop=True))
grouped["share"] = grouped["importance"] / grouped["importance"].sum()
display(grouped.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.6))
g = grouped.sort_values("importance")
colors = ["#1f4e79" if i >= len(g) - 3 else "#9db8d2" for i in range(len(g))]
bars = ax.barh(g["base"], g["importance"], color=colors, height=0.66)
for b, v, s in zip(bars, g["importance"], g["share"]):
    ax.text(v + 0.005, b.get_y() + b.get_height() / 2, f"{v:.3f}  ({s:.0%})",
            va="center", fontsize=8.5, color="#333333")
ax.set_xlabel("summed feature_importances_")
ax.set_title("A. Which variables drive the survival prediction?")
ax.set_xlim(0, g["importance"].max() * 1.28)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
save(fig, "A_feature_importance.png")

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.4))
plot_df = imp.sort_values("importance")
colors = ["#1f4e79" if i < 5 else "#9db8d2" for i in range(len(plot_df))][::-1]
bars = ax.barh(plot_df["feature"], plot_df["importance"], color=colors, height=0.68)
for b, v in zip(bars, plot_df["importance"]):
    ax.text(v + 0.004, b.get_y() + b.get_height() / 2, f"{v:.3f}", va="center", fontsize=8.5)
ax.set_xlabel("Mean decrease in impurity")
ax.set_title("A. Feature importance at the encoded-column level")
ax.set_xlim(0, plot_df["importance"].max() * 1.18)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
save(fig, "A_feature_importance_detail.png")

print("Sex + Title together:", f"{grouped.loc[grouped.base.isin(['Sex','Title']),'importance'].sum():.1%}",
      "of the total importance")

## B. Age imputation learned from the training rows only

Compute the median on the **training subset**, then use that one number for the missing ages in both
subsets. The table shows why the source matters: the three subsets do not share a median.

In [ ]:
median_train = X_tr["Age"].median()
median_va = X_va["Age"].median()
median_test = X_test_kaggle["Age"].median()
median_all = X["Age"].median()
median_pooled = pd.concat([X_tr, X_va, X_test_kaggle])["Age"].median()

age_tbl = pd.DataFrame([
    ("training rows (80 % split)", len(X_tr), X_tr["Age"].isna().sum(), median_train),
    ("validation rows (20 % split)", len(X_va), X_va["Age"].isna().sum(), median_va),
    ("Kaggle test.csv", len(X_test_kaggle), X_test_kaggle["Age"].isna().sum(), median_test),
    ("all 891 training rows", len(X), X["Age"].isna().sum(), median_all),
    ("train + test pooled (leaky)", len(X_tr) + len(X_va) + len(X_test_kaggle),
     pd.concat([X_tr, X_va, X_test_kaggle])["Age"].isna().sum(), median_pooled),
], columns=["subset", "n_rows", "n_missing_age", "median_age"])
display(age_tbl.round(2))

# the rule: one value, learned on training rows, applied to both subsets
X_tr_f, X_va_f, X_test_f = X_tr.copy(), X_va.copy(), X_test_kaggle.copy()
for frame in (X_tr_f, X_va_f, X_test_f):
    frame["Age"] = frame["Age"].fillna(median_train)

pipe_b = build_pipeline(rf())
pipe_b.fit(X_tr_f, y_tr)
pred_va = pipe_b.predict(X_va_f)
print(f"imputation value {median_train:.1f} | validation accuracy {accuracy_score(y_va, pred_va):.4f}")

In [ ]:
# proof: the pipeline's SimpleImputer learns exactly the same number from X_tr
pipe_b2 = build_pipeline(rf())
pipe_b2.fit(X_tr, y_tr)
learned = pipe_b2.named_steps["prep"].named_transformers_["num"].statistics_[0]
print(f"SimpleImputer learned median: {learned:.1f}")
print("identical validation predictions:", np.array_equal(pred_va, pipe_b2.predict(X_va)))

# what happens when the value is learned from the wrong rows
rows = []
for label, med in [("train median (correct)", median_train),
                   ("validation median (leak)", median_va),
                   ("test.csv median (leak)", median_test),
                   ("pooled median (leak)", median_pooled)]:
    a, b = X_tr.copy(), X_va.copy()
    a["Age"], b["Age"] = a["Age"].fillna(med), b["Age"].fillna(med)
    p = build_pipeline(rf()); p.fit(a, y_tr)
    rows.append({"imputation_value_source": label, "value_used": round(med, 2),
                 "rows_with_different_age": int((a["Age"] != X_tr_f["Age"]).sum() +
                                                (b["Age"] != X_va_f["Age"]).sum()),
                 "validation_accuracy": round(accuracy_score(y_va, p.predict(b)), 4)})
display(pd.DataFrame(rows))

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 3.9))
bins = np.arange(0, 85, 4)
ax.hist(X_tr["Age"].dropna(), bins=bins, color="#1f4e79", alpha=0.75, label="training rows", edgecolor="white")
ax.hist(X_va["Age"].dropna(), bins=bins, color="#8a8f98", alpha=0.55, label="validation rows", edgecolor="white")
ax.axvline(median_train, color="#2e7d32", lw=2.2, label=f"training median = {median_train:.0f}")
ax.axvline(median_test, color="#b3261e", lw=1.8, ls="--", label=f"test.csv median = {median_test:.0f}")
ax.set_xlabel("Age"); ax.set_ylabel("passengers")
ax.set_title("B. One imputation value, learned from the training subset only")
ax.legend(frameon=False, fontsize=8.5)
fig.tight_layout()
save(fig, "B_age_imputation.png")

## C. Classification report and confusion matrix

Hold-out = the 179 passengers the model never saw. Positive class = **Survived**.

In [ ]:
LABELS = ["Did not survive", "Survived"]
acc = accuracy_score(y_va, pred_va)
print(f"hold-out accuracy: {acc:.4f}\n")
print(classification_report(y_va, pred_va, target_names=LABELS, digits=4))

cm = confusion_matrix(y_va, pred_va)
display(pd.DataFrame(cm, index=[f"actual: {l}" for l in LABELS],
                     columns=[f"predicted: {l}" for l in LABELS]))

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cmap = LinearSegmentedColormap.from_list("blues", ["#ffffff", "#c9dcee", "#1f4e79"])
fig, ax = plt.subplots(figsize=(5.6, 4.6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
disp.plot(ax=ax, cmap=cmap, colorbar=False, values_format="d")
for text, (i, j) in zip(ax.texts, [(0, 0), (0, 1), (1, 0), (1, 1)]):
    text.set_text(f"{cm[i, j]}\n({cm[i, j] / cm.sum():.1%})")
    text.set_fontsize(10.5)
    text.set_color("white" if cm[i, j] > cm.max() * 0.55 else "#1a1a1a")
ax.set_title(f"C. Confusion matrix (hold-out, accuracy {acc:.1%})", fontsize=11)
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
fig.tight_layout()
save(fig, "C_confusion_matrix.png")

## D. False positives and false negatives

With **Survived** as the positive class:

* **FP** — predicted *survived*, actually died: a survival opportunity that was not there (a false alarm).
* **FN** — predicted *died*, actually survived: a survivor the model gave up on (a missed case).

The counts, the rates and the per-class recall answer the question "which class does the model find
more reliably?".

In [ ]:
tn, fp, fn, tp = cm.ravel()
err = pd.DataFrame([
    ("True negative (TN)", "Did not survive", "Did not survive", tn),
    ("False positive (FP) - Type I", "Did not survive", "Survived", fp),
    ("False negative (FN) - Type II", "Survived", "Did not survive", fn),
    ("True positive (TP)", "Survived", "Survived", tp),
], columns=["outcome", "actual", "predicted", "count"])
err["share_of_holdout"] = (err["count"] / len(y_va)).round(4)
display(err)

perf = pd.DataFrame([
    ("Did not survive (0)", precision_score(y_va, pred_va, pos_label=0),
     recall_score(y_va, pred_va, pos_label=0), f1_score(y_va, pred_va, pos_label=0), int(tn + fp)),
    ("Survived (1)", precision_score(y_va, pred_va), recall_score(y_va, pred_va),
     f1_score(y_va, pred_va), int(tp + fn)),
], columns=["class", "precision", "recall", "f1", "support"])
display(perf.round(4))

print(f"FP rate among non-survivors: {fp / (tn + fp):.3f}")
print(f"FN rate among survivors    : {fn / (tp + fn):.3f}")
print("higher recall              :", "Did not survive" if tn / (tn + fp) > tp / (tp + fn) else "Survived")

In [ ]:
va = X_va.copy()
va["actual"], va["predicted"] = y_va.values, pred_va
va["prob_survived"] = pipe_b.predict_proba(X_va_f)[:, 1].round(3)
va["Name"], va["Ticket"] = train.loc[va.index, "Name"].values, train.loc[va.index, "Ticket"].values
cols = ["Name", "Sex", "Age", "Pclass", "Fare", "FamilySize", "Title", "actual", "predicted", "prob_survived"]

print("FALSE POSITIVES - model was confident, passenger died")
display(va[(va.actual == 0) & (va.predicted == 1)].sort_values("prob_survived", ascending=False)[cols].head(6))

print("FALSE NEGATIVES - model was confident, passenger survived")
display(va[(va.actual == 1) & (va.predicted == 0)].sort_values("prob_survived")[cols].head(6))

**Reading the two error types.** The false positives are almost all **women travelling alone in 3rd
class** (`Miss`, low fare, no family): the forest has learned "female ⇒ survives" very strongly and has
no feature that says this particular woman was below deck. The false negatives are almost all **men in
2nd/3rd class with a cheap ticket and no family** — the group with the lowest survival rate (~14 %) —
and the model gives them a probability near 0.00–0.06, so it is applying a rule, not hesitating.

**Which class has the higher recall?** *Did not survive* — the model recovers ~87 % of the passengers who
died but only ~70 % of those who lived, because 61.6 % of the training rows are non-survivors and the
female-survivor rule is not strong enough to catch the surviving men. If survivor recall mattered more
than accuracy, the levers are a lower decision threshold (`predict_proba(...)[:, 1] > 0.4`),
`class_weight="balanced"`, or optimising F1 on the positive class — each trades accuracy for recall,
which the competition metric does not reward.

## E. Decision trees with `max_depth` = 1, 2, 4, 8

`random_state=42` for all four. "Actual depth" is `tree.get_depth()`, and one leaf = one rule.

In [ ]:
depth_rows, trees = [], {}
for d in (1, 2, 4, 8):
    pipe_e = build_pipeline(DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE))
    pipe_e.fit(X_tr, y_tr)
    fitted = pipe_e.named_steps["model"]
    trees[d] = (pipe_e, fitted)
    depth_rows.append({
        "max_depth": d,
        "actual_tree_depth": fitted.get_depth(),
        "n_leaves": fitted.get_n_leaves(),
        "n_nodes": fitted.tree_.node_count,
        "train_accuracy": round(accuracy_score(y_tr, pipe_e.predict(X_tr)), 4),
        "validation_accuracy": round(accuracy_score(y_va, pipe_e.predict(X_va)), 4),
    })

depth_tbl = pd.DataFrame(depth_rows)
depth_tbl["train_val_gap"] = (depth_tbl.train_accuracy - depth_tbl.validation_accuracy).round(4)
display(depth_tbl)

In [ ]:
rule_names = [n.split("__")[-1] for n in trees[1][0].named_steps["prep"].get_feature_names_out()]

for d in (1, 2):
    fitted = trees[d][1]
    print(f"--- max_depth = {d}: actual depth {fitted.get_depth()}, {fitted.get_n_leaves()} rules ---")
    print(export_text(fitted, feature_names=rule_names, decimals=2, show_weights=True))

# the full rule set for all four depths is long: depth 8 alone has 57 rules
print("rules per depth:", {d: t[1].get_n_leaves() for d, t in trees.items()})

In [ ]:
for d in (1, 2):
    fitted = trees[d][1]
    fig, ax = plt.subplots(figsize=(11 if d == 2 else 7.5, 4.2 if d == 2 else 3.2))
    plot_tree(fitted, feature_names=rule_names, class_names=LABELS, filled=True, rounded=True,
              fontsize=9, ax=ax, precision=2)
    ax.set_title(f"E. Decision tree with max_depth = {d}")
    fig.tight_layout()
    save(fig, f"E_tree_depth_{d}.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0))
axes[0].plot(depth_tbl.max_depth, depth_tbl.train_accuracy, "o-", color="#1f4e79", lw=2, label="training accuracy")
axes[0].plot(depth_tbl.max_depth, depth_tbl.validation_accuracy, "s--", color="#b3261e", lw=2, label="validation accuracy")
for _, r in depth_tbl.iterrows():
    axes[0].annotate(f"{r.train_accuracy:.3f}", (r.max_depth, r.train_accuracy), textcoords="offset points",
                     xytext=(6, 7), fontsize=8, color="#1f4e79")
    axes[0].annotate(f"{r.validation_accuracy:.3f}", (r.max_depth, r.validation_accuracy), textcoords="offset points",
                     xytext=(6, -13), fontsize=8, color="#b3261e")
axes[0].set_xticks(depth_tbl.max_depth); axes[0].set_ylim(0.745, 0.94)
axes[0].set_xlabel("max_depth"); axes[0].set_ylabel("accuracy")
axes[0].set_title("Accuracy vs. tree depth")
axes[0].legend(frameon=False, fontsize=8.5, loc="upper left")
axes[0].grid(alpha=0.25)

bars = axes[1].bar(depth_tbl.max_depth.astype(str), depth_tbl.n_leaves, color="#1f4e79", width=0.55)
for b, v in zip(bars, depth_tbl.n_leaves):
    axes[1].text(b.get_x() + b.get_width() / 2, v + 1.5, str(v), ha="center", fontsize=9)
axes[1].set_xlabel("max_depth"); axes[1].set_ylabel("leaves (= rules)")
axes[1].set_title("Rule count grows fast")
axes[1].grid(axis="y", alpha=0.25)
fig.suptitle("E. Limiting depth trades rules for generalisation", fontweight="bold")
fig.tight_layout()
save(fig, "E_depth_comparison.png")

**What changes as depth grows.** Depth 1 asks one question and the tree does **not** split on `Sex` — it
splits on `Title_Mr`, a *better* version of "male" because it excludes `Master` (boys, who mostly
survived): adult man ⇒ dies, otherwise ⇒ lives. Depth 2 adds the first *conditional* rule (for non-Mr
passengers ask about class, for Mr passengers ask about cabin) — that is what a second level buys.
Depth 4 reaches 16 rules and starts using `Age`, `Fare` and `FamilySize`. Depth 8 has 57 rules,
training accuracy rises to 0.92 while validation falls back below the depth-4 tree: the extra rules
describe individual passengers rather than a pattern.

## F. Final model and the submission

Configuration picked with a 5-fold CV grid over `min_samples_leaf` × `max_features` × `class_weight`.
Following the hint, the tuned forest is refitted on **all 891 training rows** before predicting the
418 test passengers.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(build_pipeline(rf()), X, y, cv=cv, n_jobs=-1)
print(f"5-fold CV      : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
cv_repeated = cross_val_score(build_pipeline(rf()), X, y, cv=cv_rep, n_jobs=-1)
print(f"5x5 repeated CV: {cv_repeated.mean():.4f} +/- {cv_repeated.std():.4f}")

comparison = pd.DataFrame([
    ("Sex rule (depth-1 equivalent)", cross_val_score(
        build_pipeline(DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)), X, y, cv=cv).mean()),
    ("Decision Tree, max_depth=4", cross_val_score(
        build_pipeline(DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)), X, y, cv=cv).mean()),
    ("Random Forest, defaults", cross_val_score(
        build_pipeline(RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE)), X, y, cv=cv).mean()),
    ("Random Forest, tuned", cv_scores.mean()),
], columns=["model", "cv_accuracy"])
display(comparison.round(4))

In [ ]:
final_pipe = build_pipeline(rf())
final_pipe.fit(X, y)          # all 891 training rows, as the hint suggests

X_test_final = X_test_kaggle.copy()
X_test_final["Age"] = X_test_final["Age"].fillna(X["Age"].median())
pred = final_pipe.predict(X_test_final)
prob = final_pipe.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": pred.astype(int)})
submission.to_csv(OUT / "submission.csv", index=False)

checks = {
    "columns": list(submission.columns) == ["PassengerId", "Survived"],
    "418 rows": len(submission) == 418,
    "ids match test.csv": submission.PassengerId.equals(test.PassengerId),
    "no missing": not submission.isna().any().any(),
    "binary": set(submission.Survived.unique()) <= {0, 1},
}
print(checks)
print("predicted survival rate:", round(submission.Survived.mean(), 4))
print("saved to:", (OUT / "submission.csv").resolve())
submission.head()

### How to submit this notebook to the leaderboard

1. **Save Version → Save & Run All (Commit)** — this executes the notebook top to bottom and produces
   `submission.csv` in the output.
2. In the right-hand **Output** panel (or the version viewer), open the `...` menu next to
   `submission.csv` → **Submit to competition** → *Titanic*.
3. The score appears on the competition leaderboard and as a badge on this notebook.
4. **Share → Visibility → Public** (or keep it private and add your professor as a *Can view*
   collaborator) and send the notebook URL:
   `https://www.kaggle.com/code/<your-username>/<notebook-slug>`

That single link contains the code, the outputs and the leaderboard score — which is what an instructor
needs to check the exercise.